# FASE 3: Predicción y generación de resultados para Power BI

**Proyecto:** Sistema de alerta temprana de accidentes de tránsito en Cuenca mediante una Red Neuronal Artificial.

Esta fase carga los artefactos de las fases anteriores, permite ingresar nuevos datos y genera una salida lista para Power BI.

## 1. Archivos necesarios

Coloca en la misma carpeta:

- `modelo_riesgo_ubicacion_cuenca.keras`
- `datos_fase1_riesgo_ubicacion.pkl`
- `resumen_historico_powerbi.csv`

## 2. Importación de librerías

In [85]:
from pathlib import Path
import pickle
import unicodedata


In [86]:
import numpy as np
import pandas as pd
from tensorflow.keras.models import load_model

## 3. Configuración y verificación de rutas

In [87]:
CARPETA_PROYECTO = Path.cwd()
RUTA_MODELO = CARPETA_PROYECTO / "modelo_riesgo_ubicacion_cuenca.keras"
RUTA_DATOS = CARPETA_PROYECTO / "datos_fase1_riesgo_ubicacion.pkl"
RUTA_RESUMEN = CARPETA_PROYECTO / "resumen_historico_powerbi.csv"

faltantes = [p.name for p in [RUTA_MODELO, RUTA_DATOS, RUTA_RESUMEN] if not p.exists()]
if faltantes:
    raise FileNotFoundError("Faltan archivos: " + ", ".join(faltantes))

print("Todos los archivos fueron encontrados.")

Todos los archivos fueron encontrados.


## 4. Carga del modelo, preprocesador y resumen histórico

In [88]:
modelo = load_model(RUTA_MODELO)
with open(RUTA_DATOS, "rb") as archivo:
    datos_fase1 = pickle.load(archivo)

preprocesador = datos_fase1["preprocesador"]
columnas_entrada = datos_fase1["columnas_entrada"]
resumen_historico = pd.read_csv(RUTA_RESUMEN)

print("Modelo cargado correctamente.")
print("Columnas esperadas:", columnas_entrada)
print("Registros históricos:", len(resumen_historico))

Modelo cargado correctamente.
Columnas esperadas: ['MES', 'DIA_SEMANA', 'HORA_NUM', 'RANGO_HORARIO', 'PARROQUIA', 'ZONA', 'FERIADO']
Registros históricos: 870


## 5. Normalización del resumen histórico

In [89]:
def normalizar_texto(valor):
    if pd.isna(valor):
        return pd.NA
    texto = str(valor).strip().upper()
    texto = unicodedata.normalize("NFD", texto)
    return "".join(c for c in texto if unicodedata.category(c) != "Mn")

resumen_historico.columns = resumen_historico.columns.astype(str).str.strip().str.upper()
for columna in resumen_historico.select_dtypes(include=["object"]).columns:
    resumen_historico[columna] = resumen_historico[columna].apply(normalizar_texto).astype("string")

for columna in ["MES", "DIA_SEMANA", "NUM_ACCIDENTES", "PROMEDIO_LESIONADOS", "PROMEDIO_FALLECIDOS", "LATITUD_PROMEDIO", "LONGITUD_PROMEDIO"]:
    if columna in resumen_historico.columns:
        resumen_historico[columna] = pd.to_numeric(resumen_historico[columna], errors="coerce")

resumen_historico.head()

C:\Users\avila\AppData\Local\Temp\ipykernel_47568\861367524.py:9: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for columna in resumen_historico.select_dtypes(include=["object"]).columns:


,MES,DIA_SEMANA,RANGO_HORARIO,PARROQUIA,ZONA,FERIADO,NUM_ACCIDENTES,CAUSA_MAS_FRECUENTE,SINIESTRO_MAS_FRECUENTE,VEHICULO_MAS_FRECUENTE,PROMEDIO_LESIONADOS,PROMEDIO_FALLECIDOS,PROMEDIO_VICTIMAS,LATITUD_PROMEDIO,LONGITUD_PROMEDIO
0,1,0,MADRUGADA,CUENCA,URBANA,SI,2,"CONDUCE BAJO LA INFLUENCIA DE ALCOHOL, SUSTANC...",CHOQUE POSTERIOR,MOTOCICLETA,0.50,0.0,0.50,-2.904572,-78.979707
1,1,0,MANANA,CUENCA,URBANA,NO,2,CONDUCIR DESATENTO A LAS CONDICIONES DE TRANSI...,ATROPELLOS,AUTOMOVIL,1.00,0.0,1.00,-2.907090,-78.999710
2,1,0,MANANA,CUENCA,URBANA,SI,1,CONDUCIR DESATENTO A LAS CONDICIONES DE TRANSI...,PERDIDA DE CARRIL,AUTOMOVIL,0.00,0.0,0.00,-2.919688,-79.004881
3,1,0,MANANA,VALLE,RURAL,NO,1,CONDUCIR DESATENTO A LAS CONDICIONES DE TRANSI...,PERDIDA DE PISTA,AUTOMOVIL,1.00,0.0,1.00,-2.930489,-78.967382
4,1,0,NOCHE,CUENCA,URBANA,NO,3,"CONDUCE BAJO LA INFLUENCIA DE ALCOHOL, SUSTANC...",CHOQUE LATERAL,AUTOMOVIL,1.33,0.0,1.33,-2.891137,-78.986230


## 6. Funciones auxiliares

In [90]:
def validar_hora(hora):
    hora = int(hora)
    if not 0 <= hora <= 23:
        raise ValueError("La hora debe estar entre 0 y 23.")
    return hora

def clasificar_rango_horario(hora):
    if hora <= 5:
        return "MADRUGADA"
    if hora <= 11:
        return "MANANA"
    if hora <= 17:
        return "TARDE"
    return "NOCHE"

def clasificar_nivel_riesgo(p):
    if p >= 70:
        return "ALTO"
    if p >= 40:
        return "MEDIO"
    return "BAJO"

def color_riesgo(nivel):
    return {"BAJO": "VERDE", "MEDIO": "AMARILLO", "ALTO": "ROJO"}[nivel]

## 7. Valores válidos para la interfaz

In [91]:
parroquias_disponibles = sorted(resumen_historico["PARROQUIA"].dropna().astype(str).unique().tolist())
zonas_disponibles = sorted(resumen_historico["ZONA"].dropna().astype(str).unique().tolist())

print("Parroquias:", len(parroquias_disponibles))
print("Zonas:", zonas_disponibles)

Parroquias: 22
Zonas: ['RURAL', 'URBANA']


## 8. Consulta histórica

In [92]:
def seleccionar_fila(datos):
    if datos.empty:
        return None
    if "NUM_ACCIDENTES" in datos.columns:
        return datos.sort_values("NUM_ACCIDENTES", ascending=False).iloc[0]
    return datos.iloc[0]

def consultar_historial(mes, dia_semana, rango_horario, parroquia, zona, feriado):
    filtros = [
        (resumen_historico[(resumen_historico["MES"] == mes) & (resumen_historico["DIA_SEMANA"] == dia_semana) & (resumen_historico["RANGO_HORARIO"] == rango_horario) & (resumen_historico["PARROQUIA"] == parroquia) & (resumen_historico["ZONA"] == zona) & (resumen_historico["FERIADO"] == feriado)], "Coincidencia exacta"),
        (resumen_historico[(resumen_historico["RANGO_HORARIO"] == rango_horario) & (resumen_historico["PARROQUIA"] == parroquia) & (resumen_historico["ZONA"] == zona)], "Coincidencia aproximada por lugar y horario"),
        (resumen_historico[(resumen_historico["PARROQUIA"] == parroquia) & (resumen_historico["ZONA"] == zona)], "Referencia por parroquia y zona"),
        (resumen_historico[resumen_historico["PARROQUIA"] == parroquia], "Referencia general de la parroquia")
    ]
    for datos, descripcion in filtros:
        fila = seleccionar_fila(datos)
        if fila is not None:
            return fila, descripcion
    return None, "Sin antecedentes históricos suficientes"

## 9. Función principal de predicción

In [93]:
def predecir_riesgo(
    fecha,
    hora,
    parroquia,
    zona,
    feriado
):
    fecha = pd.to_datetime(fecha, errors="coerce")

    if pd.isnull(fecha):
        raise ValueError(
            "La fecha no es válida. Usa el formato AAAA-MM-DD."
        )

    hora = validar_hora(hora)
    parroquia = normalizar_texto(parroquia)
    zona = normalizar_texto(zona)
    feriado = normalizar_texto(feriado)

    if parroquia not in parroquias_disponibles:
        raise ValueError(
            f"La parroquia {parroquia} no está disponible."
        )

    if zona not in zonas_disponibles:
        raise ValueError(
            f"Zona inválida. Opciones: {zonas_disponibles}"
        )

    if feriado not in ["SI", "NO"]:
        raise ValueError(
            "Feriado debe ser SI o NO."
        )

    mes = int(fecha.month)
    dia_semana = int(fecha.dayofweek)
    rango_horario = clasificar_rango_horario(hora)

    nuevo = pd.DataFrame({
        "MES": [mes],
        "DIA_SEMANA": [dia_semana],
        "HORA_NUM": [hora],
        "RANGO_HORARIO": [rango_horario],
        "PARROQUIA": [parroquia],
        "ZONA": [zona],
        "FERIADO": [feriado]
    })

    print("Columnas creadas:")
    print(nuevo.columns.tolist())

    print("\nColumnas esperadas:")
    print(columnas_entrada)

    nuevo = nuevo[columnas_entrada]

    nuevo_transformado = preprocesador.transform(nuevo)

    prediccion_escalada = modelo.predict(
        nuevo_transformado,
        verbose=0
    ).reshape(-1)[0]

    riesgo_porcentaje = float(
        np.clip(prediccion_escalada * 100, 0, 100)
    )

    nivel_riesgo = clasificar_nivel_riesgo(
        riesgo_porcentaje
    )

    historial, referencia = consultar_historial(
        mes=mes,
        dia_semana=dia_semana,
        rango_horario=rango_horario,
        parroquia=parroquia,
        zona=zona,
        feriado=feriado
    )

    resultado = {
        "FECHA": fecha.strftime("%Y-%m-%d"),
        "HORA": hora,
        "RANGO_HORARIO": rango_horario,
        "PARROQUIA": parroquia,
        "ZONA": zona,
        "FERIADO": feriado,
        "RIESGO_PORCENTAJE": round(
            riesgo_porcentaje,
            2
        ),
        "NIVEL_RIESGO": nivel_riesgo,
        "REFERENCIA_HISTORICA": referencia
    }

    if historial is not None:
        resultado.update({
            "CAUSA_MAS_FRECUENTE": historial.get(
                "CAUSA_MAS_FRECUENTE",
                "SIN INFORMACION"
            ),
            "SINIESTRO_MAS_FRECUENTE": historial.get(
                "SINIESTRO_MAS_FRECUENTE",
                "SIN INFORMACION"
            ),
            "VEHICULO_MAS_FRECUENTE": historial.get(
                "VEHICULO_MAS_FRECUENTE",
                "SIN INFORMACION"
            ),
            "PROMEDIO_LESIONADOS": historial.get(
                "PROMEDIO_LESIONADOS",
                0
            ),
            "PROMEDIO_FALLECIDOS": historial.get(
                "PROMEDIO_FALLECIDOS",
                0
            ),
            "LATITUD": historial.get(
                "LATITUD_PROMEDIO",
                np.nan
            ),
            "LONGITUD": historial.get(
                "LONGITUD_PROMEDIO",
                np.nan
            )
        })

    return resultado

### 9.1 FUNCION PARA VISUALIZAR LOS NIVELES: BAJO, MEDIO, ALTO

In [94]:
from itertools import product

def buscar_ejemplos_por_nivel(cantidad_por_nivel=5):
    registros = []

    fechas_prueba = pd.date_range(
        start="2025-01-01",
        end="2025-12-31",
        freq="14D"
    )

    horas_prueba = [1, 7, 10, 13, 16, 18, 20, 22]
    feriados_prueba = ["SI", "NO"]

    # Crear todas las combinaciones sin predecir todavía
    for fecha, hora, parroquia, zona, feriado in product(
        fechas_prueba,
        horas_prueba,
        parroquias_disponibles,
        zonas_disponibles,
        feriados_prueba
    ):
        registros.append({
            "FECHA": fecha.strftime("%Y-%m-%d"),
            "MES": fecha.month,
            "DIA_SEMANA": fecha.dayofweek,
            "HORA_NUM": hora,
            "RANGO_HORARIO": clasificar_rango_horario(hora),
            "PARROQUIA": parroquia,
            "ZONA": zona,
            "FERIADO": feriado
        })

    candidatos = pd.DataFrame(registros)

    # Mantener únicamente las columnas esperadas por el modelo
    X_nuevos = candidatos[columnas_entrada]

    # Transformar todo de una sola vez
    X_nuevos_transformados = preprocesador.transform(X_nuevos)

    # Predecir todo en lote
    predicciones_escaladas = modelo.predict(
        X_nuevos_transformados,
        batch_size=256,
        verbose=0
    ).reshape(-1)

    candidatos["RIESGO_PORCENTAJE"] = np.clip(
        predicciones_escaladas * 100,
        0,
        100
    ).round(2)

    candidatos["NIVEL_RIESGO"] = candidatos[
        "RIESGO_PORCENTAJE"
    ].apply(clasificar_nivel_riesgo)

    encontrados = {}

    for nivel in ["BAJO", "MEDIO", "ALTO"]:
        ejemplos_nivel = candidatos[
            candidatos["NIVEL_RIESGO"] == nivel
        ].copy()

        if nivel == "BAJO":
            ejemplos_nivel = ejemplos_nivel.sort_values(
                "RIESGO_PORCENTAJE",
                ascending=True
            )

        elif nivel == "MEDIO":
            ejemplos_nivel["DISTANCIA_CENTRO"] = (
                ejemplos_nivel["RIESGO_PORCENTAJE"] - 55
            ).abs()

            ejemplos_nivel = ejemplos_nivel.sort_values(
                "DISTANCIA_CENTRO"
            ).drop(columns="DISTANCIA_CENTRO")

        else:
            ejemplos_nivel = ejemplos_nivel.sort_values(
                "RIESGO_PORCENTAJE",
                ascending=False
            )

        encontrados[nivel] = ejemplos_nivel.head(
            cantidad_por_nivel
        ).to_dict("records")

    return encontrados, candidatos

In [95]:
ejemplos_encontrados, candidatos = buscar_ejemplos_por_nivel(
    cantidad_por_nivel=5
)

for nivel, ejemplos in ejemplos_encontrados.items():
    print(
        f"{nivel}: {len(ejemplos)} ejemplos encontrados"
    )

BAJO: 5 ejemplos encontrados
MEDIO: 5 ejemplos encontrados
ALTO: 4 ejemplos encontrados


In [96]:
lista_ejemplos = []

for nivel, ejemplos in ejemplos_encontrados.items():
    lista_ejemplos.extend(ejemplos)

tabla_ejemplos_niveles = pd.DataFrame(lista_ejemplos)

columnas_mostrar = [
    "FECHA",
    "HORA_NUM",
    "RANGO_HORARIO",
    "PARROQUIA",
    "ZONA",
    "FERIADO",
    "RIESGO_PORCENTAJE",
    "NIVEL_RIESGO"
]

tabla_ejemplos_niveles[columnas_mostrar]

,FECHA,HORA_NUM,RANGO_HORARIO,PARROQUIA,ZONA,FERIADO,RIESGO_PORCENTAJE,NIVEL_RIESGO
0,2025-01-29,1,MADRUGADA,LLACAO,RURAL,NO,6.800000,BAJO
1,2025-01-15,1,MADRUGADA,LLACAO,RURAL,NO,6.800000,BAJO
2,2025-01-01,1,MADRUGADA,LLACAO,RURAL,NO,6.800000,BAJO
3,2025-05-21,1,MADRUGADA,LLACAO,RURAL,NO,6.840000,BAJO
4,2025-05-07,1,MADRUGADA,LLACAO,RURAL,NO,6.840000,BAJO
5,2025-07-02,7,MANANA,CUENCA,URBANA,NO,54.700001,MEDIO
6,2025-07-16,7,MANANA,CUENCA,URBANA,NO,54.700001,MEDIO
7,2025-07-30,7,MANANA,CUENCA,URBANA,NO,54.700001,MEDIO
8,2025-09-10,7,MANANA,CUENCA,URBANA,NO,55.730000,MEDIO
9,2025-09-24,7,MANANA,CUENCA,URBANA,NO,55.730000,MEDIO


## 11. Entrada manual del usuario

#### EJEMPLO DE NIVEL BAJO

In [97]:
def ejecutar_sistema():
    print("=" * 60)
    print("ENTRADA DEL USUARIO")
    print("=" * 60)

    fecha_usuario = input(
        "Ingrese la fecha (AAAA-MM-DD): "
    ).strip()

    hora_usuario = int(
        input("Ingrese la hora (0-23): ")
    )

    print("\nParroquias disponibles:")
    for indice, parroquia in enumerate(
        parroquias_disponibles,
        start=1
    ):
        print(f"{indice}. {parroquia}")

    opcion_parroquia = int(
        input("\nSeleccione el número de la parroquia: ")
    )

    if not 1 <= opcion_parroquia <= len(
        parroquias_disponibles
    ):
        raise ValueError(
            "La opción de parroquia no es válida."
        )

    parroquia_usuario = parroquias_disponibles[
        opcion_parroquia - 1
    ]

    zona_usuario = input(
        "Ingrese la zona (URBANA/RURAL): "
    ).strip().upper()

    feriado_usuario = input(
        "¿Es feriado? (SI/NO): "
    ).strip().upper()

    resultado = predecir_riesgo(
        fecha=fecha_usuario,
        hora=hora_usuario,
        parroquia=parroquia_usuario,
        zona=zona_usuario,
        feriado=feriado_usuario
    )

    print()
    print("=" * 60)
    print("SISTEMA DE ALERTA TEMPRANA DE ACCIDENTES")
    print("=" * 60)

    print(f"Fecha: {resultado['FECHA']}")
    print(f"Hora: {resultado['HORA']:02d}:00")
    print(
        f"Rango horario: "
        f"{resultado['RANGO_HORARIO']}"
    )
    print(f"Parroquia: {resultado['PARROQUIA']}")
    print(f"Zona: {resultado['ZONA']}")
    print(f"Feriado: {resultado['FERIADO']}")

    print()

    print(
        f"Riesgo estimado: "
        f"{resultado['RIESGO_PORCENTAJE']:.2f}%"
    )
    print(
        f"Nivel de riesgo: "
        f"{resultado['NIVEL_RIESGO']}"
    )

    print()

    print(
        f"Causa más frecuente: "
        f"{resultado['CAUSA_MAS_FRECUENTE']}"
    )
    print(
        f"Siniestro más frecuente: "
        f"{resultado['SINIESTRO_MAS_FRECUENTE']}"
    )
    print(
        f"Vehículo más frecuente: "
        f"{resultado['VEHICULO_MAS_FRECUENTE']}"
    )

    print()

    print(
        f"Promedio lesionados: "
        f"{resultado['PROMEDIO_LESIONADOS']}"
    )
    print(
        f"Promedio fallecidos: "
        f"{resultado['PROMEDIO_FALLECIDOS']}"
    )

    print()

    print(f"Latitud: {resultado['LATITUD']}")
    print(f"Longitud: {resultado['LONGITUD']}")

    print()

    print(
        f"Referencia histórica: "
        f"{resultado['REFERENCIA_HISTORICA']}"
    )

    print("=" * 60)

    return resultado

resultado_final = ejecutar_sistema()

ENTRADA DEL USUARIO

Parroquias disponibles:
1. BANOS (CUENCA)
2. CHAUCHA
3. CHECA (JIDCAY)
4. CHIQUINTAD
5. CUENCA
6. CUMBE
7. LLACAO
8. MOLLETURO
9. NULTI
10. OCTAVIO CORDERO PALACIOS
11. PACCHA (CUENCA)
12. QUINGEO
13. RICAURTE (CUENCA)
14. SAN JOAQUIN
15. SANTA ANA
16. SAYAUSI
17. SIDCAY
18. SININCAY
19. TARQUI (CUENCA)
20. TURI
21. VALLE
22. VICTORIA DEL PORTETE
Columnas creadas:
['MES', 'DIA_SEMANA', 'HORA_NUM', 'RANGO_HORARIO', 'PARROQUIA', 'ZONA', 'FERIADO']

Columnas esperadas:
['MES', 'DIA_SEMANA', 'HORA_NUM', 'RANGO_HORARIO', 'PARROQUIA', 'ZONA', 'FERIADO']

SISTEMA DE ALERTA TEMPRANA DE ACCIDENTES
Fecha: 2025-12-02
Hora: 18:00
Rango horario: NOCHE
Parroquia: CUENCA
Zona: RURAL
Feriado: SI

Riesgo estimado: 13.46%
Nivel de riesgo: BAJO

Causa más frecuente: CONDUCE BAJO LA INFLUENCIA DE ALCOHOL, SUSTANCIAS ESTUPEFACIENTES O PSICOTROPICAS Y/O MEDICAMENTOS.
Siniestro más frecuente: ROZAMIENTOS
Vehículo más frecuente: CAMIONETA

Promedio lesionados: 0.0
Promedio fallecidos: 0.0

#### EJEMPLO DE NIVEL MEDIO

In [107]:

fecha_usuario = "2025-09-24"
hora_usuario = 7
parroquia_usuario = "CUENCA"
zona_usuario = "URBANA"
feriado_usuario = "NO"

resultado = predecir_riesgo(fecha_usuario, hora_usuario, parroquia_usuario, zona_usuario, feriado_usuario)


print("="*60)
print("SISTEMA DE ALERTA TEMPRANA DE ACCIDENTES")
print("="*60)

print(f"Fecha: {resultado['FECHA']}")
print(f"Hora: {resultado['HORA']}:00")
print(f"Rango horario: {resultado['RANGO_HORARIO']}")
print(f"Parroquia: {resultado['PARROQUIA']}")
print(f"Zona: {resultado['ZONA']}")
print(f"Feriado: {resultado['FERIADO']}")

print()

print(f"Riesgo estimado: {resultado['RIESGO_PORCENTAJE']:.2f}%")
print(f"Nivel de riesgo: {resultado['NIVEL_RIESGO']}")

print()

print(f"Causa más frecuente: {resultado['CAUSA_MAS_FRECUENTE']}")
print(f"Siniestro más frecuente: {resultado['SINIESTRO_MAS_FRECUENTE']}")
print(f"Vehículo más frecuente: {resultado['VEHICULO_MAS_FRECUENTE']}")

print()

print(f"Promedio lesionados: {resultado['PROMEDIO_LESIONADOS']}")
print(f"Promedio fallecidos: {resultado['PROMEDIO_FALLECIDOS']}")

print()

print(f"Latitud: {resultado['LATITUD']}")
print(f"Longitud: {resultado['LONGITUD']}")

print()

print(f"Referencia histórica: {resultado['REFERENCIA_HISTORICA']}")

print("="*60)

Columnas creadas:
['MES', 'DIA_SEMANA', 'HORA_NUM', 'RANGO_HORARIO', 'PARROQUIA', 'ZONA', 'FERIADO']

Columnas esperadas:
['MES', 'DIA_SEMANA', 'HORA_NUM', 'RANGO_HORARIO', 'PARROQUIA', 'ZONA', 'FERIADO']
SISTEMA DE ALERTA TEMPRANA DE ACCIDENTES
Fecha: 2025-09-24
Hora: 7:00
Rango horario: MANANA
Parroquia: CUENCA
Zona: URBANA
Feriado: NO

Riesgo estimado: 55.73%
Nivel de riesgo: MEDIO

Causa más frecuente: CONDUCIR DESATENTO A LAS CONDICIONES DE TRANSITO (CELULAR, PANTALLAS DE VIDEO, COMIDA, MAQUILLAJE O CUALQUIER OTRO ELEMENTO DISTRACTOR).
Siniestro más frecuente: CHOQUE LATERAL
Vehículo más frecuente: BUS

Promedio lesionados: 0.0
Promedio fallecidos: 0.0

Latitud: -2.885678
Longitud: -78.998287

Referencia histórica: Coincidencia exacta


#### EJEMPLO DE NIVEL ALTO

In [106]:
fecha_usuario = "2025-09-10"
hora_usuario = 16
parroquia_usuario = "CUENCA"
zona_usuario = "URBANA"
feriado_usuario = "NO"

resultado = predecir_riesgo(fecha_usuario, hora_usuario, parroquia_usuario, zona_usuario, feriado_usuario)


print("="*60)
print("SISTEMA DE ALERTA TEMPRANA DE ACCIDENTES")
print("="*60)

print(f"Fecha: {resultado['FECHA']}")
print(f"Hora: {resultado['HORA']}:00")
print(f"Rango horario: {resultado['RANGO_HORARIO']}")
print(f"Parroquia: {resultado['PARROQUIA']}")
print(f"Zona: {resultado['ZONA']}")
print(f"Feriado: {resultado['FERIADO']}")

print()

print(f"Riesgo estimado: {resultado['RIESGO_PORCENTAJE']:.2f}%")
print(f"Nivel de riesgo: {resultado['NIVEL_RIESGO']}")

print()

print(f"Causa más frecuente: {resultado['CAUSA_MAS_FRECUENTE']}")
print(f"Siniestro más frecuente: {resultado['SINIESTRO_MAS_FRECUENTE']}")
print(f"Vehículo más frecuente: {resultado['VEHICULO_MAS_FRECUENTE']}")

print()

print(f"Promedio lesionados: {resultado['PROMEDIO_LESIONADOS']}")
print(f"Promedio fallecidos: {resultado['PROMEDIO_FALLECIDOS']}")

print()

print(f"Latitud: {resultado['LATITUD']}")
print(f"Longitud: {resultado['LONGITUD']}")

print()

print(f"Referencia histórica: {resultado['REFERENCIA_HISTORICA']}")

print("="*60)

Columnas creadas:
['MES', 'DIA_SEMANA', 'HORA_NUM', 'RANGO_HORARIO', 'PARROQUIA', 'ZONA', 'FERIADO']

Columnas esperadas:
['MES', 'DIA_SEMANA', 'HORA_NUM', 'RANGO_HORARIO', 'PARROQUIA', 'ZONA', 'FERIADO']
SISTEMA DE ALERTA TEMPRANA DE ACCIDENTES
Fecha: 2025-09-10
Hora: 16:00
Rango horario: TARDE
Parroquia: CUENCA
Zona: URBANA
Feriado: NO

Riesgo estimado: 72.15%
Nivel de riesgo: ALTO

Causa más frecuente: NO RESPETAR LAS SENALES REGLAMENTARIAS DE TRANSITO. (PARE, CEDA EL PASO, LUZ ROJA DEL SEMAFORO, ETC).
Siniestro más frecuente: CHOQUE LATERAL
Vehículo más frecuente: MOTOCICLETA

Promedio lesionados: 1.3
Promedio fallecidos: 0.1

Latitud: -2.8965405
Longitud: -79.0067313

Referencia histórica: Coincidencia exacta


## 12. Resultado como DataFrame

In [108]:
resultado_df = pd.DataFrame([resultado])
resultado_df

,FECHA,HORA,RANGO_HORARIO,PARROQUIA,ZONA,FERIADO,RIESGO_PORCENTAJE,NIVEL_RIESGO,REFERENCIA_HISTORICA,CAUSA_MAS_FRECUENTE,SINIESTRO_MAS_FRECUENTE,VEHICULO_MAS_FRECUENTE,PROMEDIO_LESIONADOS,PROMEDIO_FALLECIDOS,LATITUD,LONGITUD
0,2025-09-24,7,MANANA,CUENCA,URBANA,NO,55.73,MEDIO,Coincidencia exacta,CONDUCIR DESATENTO A LAS CONDICIONES DE TRANSI...,CHOQUE LATERAL,BUS,0.0,0.0,-2.885678,-78.998287


## 13. Predicción de varios ejemplos

In [121]:
ejemplos = [
    {"fecha": "2025-01-10", "hora": 8, "parroquia": "CHAUCHA", "zona": "URBANA", "feriado": "NO"},
    {"fecha": "2025-02-15", "hora": 13, "parroquia": "CUENCA", "zona": "URBANA", "feriado": "NO"},
    {"fecha": "2025-04-18", "hora": 18, "parroquia": "LLACAO", "zona": "RURAL", "feriado": "SI"},
    {"fecha": "2025-07-15", "hora": 7, "parroquia": "BANOS (CUENCA)", "zona": "URBANA", "feriado": "NO"},
    {"fecha": "2025-12-25", "hora": 16, "parroquia": "CUENCA", "zona": "URBANA", "feriado": "NO"}
]

resultados = []
for e in ejemplos:
    try:
        resultados.append(predecir_riesgo(**e))
    except Exception as error:
        resultados.append({"FECHA_CONSULTADA": e["fecha"], "HORA": e["hora"], "PARROQUIA": e["parroquia"], "ZONA": e["zona"], "FERIADO": e["feriado"], "ERROR": str(error)})

resultados_powerbi = pd.DataFrame(resultados)
resultados_powerbi

Columnas creadas:
['MES', 'DIA_SEMANA', 'HORA_NUM', 'RANGO_HORARIO', 'PARROQUIA', 'ZONA', 'FERIADO']

Columnas esperadas:
['MES', 'DIA_SEMANA', 'HORA_NUM', 'RANGO_HORARIO', 'PARROQUIA', 'ZONA', 'FERIADO']
Columnas creadas:
['MES', 'DIA_SEMANA', 'HORA_NUM', 'RANGO_HORARIO', 'PARROQUIA', 'ZONA', 'FERIADO']

Columnas esperadas:
['MES', 'DIA_SEMANA', 'HORA_NUM', 'RANGO_HORARIO', 'PARROQUIA', 'ZONA', 'FERIADO']
Columnas creadas:
['MES', 'DIA_SEMANA', 'HORA_NUM', 'RANGO_HORARIO', 'PARROQUIA', 'ZONA', 'FERIADO']

Columnas esperadas:
['MES', 'DIA_SEMANA', 'HORA_NUM', 'RANGO_HORARIO', 'PARROQUIA', 'ZONA', 'FERIADO']
Columnas creadas:
['MES', 'DIA_SEMANA', 'HORA_NUM', 'RANGO_HORARIO', 'PARROQUIA', 'ZONA', 'FERIADO']

Columnas esperadas:
['MES', 'DIA_SEMANA', 'HORA_NUM', 'RANGO_HORARIO', 'PARROQUIA', 'ZONA', 'FERIADO']
Columnas creadas:
['MES', 'DIA_SEMANA', 'HORA_NUM', 'RANGO_HORARIO', 'PARROQUIA', 'ZONA', 'FERIADO']

Columnas esperadas:
['MES', 'DIA_SEMANA', 'HORA_NUM', 'RANGO_HORARIO', 'PARROQ

,FECHA,HORA,RANGO_HORARIO,PARROQUIA,ZONA,FERIADO,RIESGO_PORCENTAJE,NIVEL_RIESGO,REFERENCIA_HISTORICA,CAUSA_MAS_FRECUENTE,SINIESTRO_MAS_FRECUENTE,VEHICULO_MAS_FRECUENTE,PROMEDIO_LESIONADOS,PROMEDIO_FALLECIDOS,LATITUD,LONGITUD
0,2025-01-10,8,MANANA,CHAUCHA,URBANA,NO,15.44,BAJO,Referencia general de la parroquia,CONDUCIR DESATENTO A LAS CONDICIONES DE TRANSI...,PERDIDA DE PISTA,NO IDENTIFICADO,1.00,0.00,-2.875236,-79.422505
1,2025-02-15,13,TARDE,CUENCA,URBANA,NO,40.26,MEDIO,Coincidencia exacta,CONDUCIR DESATENTO A LAS CONDICIONES DE TRANSI...,ATROPELLOS,AUTOMOVIL,0.75,0.25,-2.897457,-79.005634
2,2025-04-18,18,NOCHE,LLACAO,RURAL,SI,10.23,BAJO,Coincidencia aproximada por lugar y horario,NO CEDER EL DERECHO DE VIA O PREFERENCIA DE PA...,CHOQUE FRONTAL,VEHICULO DEPORTIVO UTILITARIO,0.00,0.00,-2.849487,-78.926296
3,2025-07-15,7,MANANA,BANOS (CUENCA),URBANA,NO,20.40,BAJO,Referencia por parroquia y zona,"CONDUCE BAJO LA INFLUENCIA DE ALCOHOL, SUSTANC...",CHOQUE POSTERIOR,AUTOMOVIL,1.00,0.00,-2.937737,-79.046406
4,2025-12-25,16,TARDE,CUENCA,URBANA,NO,41.77,MEDIO,Coincidencia exacta,NO MANTENER LA DISTANCIA PRUDENCIAL CON RESPEC...,ATROPELLOS,VEHICULO DEPORTIVO UTILITARIO,0.40,0.60,-1.869319,-79.102982


## 14. Exportación para Power BI

In [122]:
resultados_powerbi.to_csv("predicciones_riesgo_powerbi.csv", index=False, encoding="utf-8-sig")
resultado_df.to_csv("ultima_prediccion_riesgo.csv", index=False, encoding="utf-8-sig")

columnas_mapa = [c for c in ["PARROQUIA", "ZONA", "LATITUD", "LONGITUD", "RIESGO_PORCENTAJE", "NIVEL_RIESGO", "COLOR_MAPA", "CAUSA_MAS_FRECUENTE", "SINIESTRO_MAS_FRECUENTE", "VEHICULO_MAS_FRECUENTE", "PROMEDIO_LESIONADOS", "PROMEDIO_FALLECIDOS"] if c in resultados_powerbi.columns]
mapa_powerbi = resultados_powerbi[columnas_mapa].copy()
mapa_powerbi.to_csv("mapa_riesgo_powerbi.csv", index=False, encoding="utf-8-sig")
    
print("Archivos generados para Power BI.")

Archivos generados para Power BI.


## 15. Vista para mapa

In [123]:
mapa_powerbi

,PARROQUIA,ZONA,LATITUD,LONGITUD,RIESGO_PORCENTAJE,NIVEL_RIESGO,CAUSA_MAS_FRECUENTE,SINIESTRO_MAS_FRECUENTE,VEHICULO_MAS_FRECUENTE,PROMEDIO_LESIONADOS,PROMEDIO_FALLECIDOS
0,CHAUCHA,URBANA,-2.875236,-79.422505,15.44,BAJO,CONDUCIR DESATENTO A LAS CONDICIONES DE TRANSI...,PERDIDA DE PISTA,NO IDENTIFICADO,1.00,0.00
1,CUENCA,URBANA,-2.897457,-79.005634,40.26,MEDIO,CONDUCIR DESATENTO A LAS CONDICIONES DE TRANSI...,ATROPELLOS,AUTOMOVIL,0.75,0.25
2,LLACAO,RURAL,-2.849487,-78.926296,10.23,BAJO,NO CEDER EL DERECHO DE VIA O PREFERENCIA DE PA...,CHOQUE FRONTAL,VEHICULO DEPORTIVO UTILITARIO,0.00,0.00
3,BANOS (CUENCA),URBANA,-2.937737,-79.046406,20.40,BAJO,"CONDUCE BAJO LA INFLUENCIA DE ALCOHOL, SUSTANC...",CHOQUE POSTERIOR,AUTOMOVIL,1.00,0.00
4,CUENCA,URBANA,-1.869319,-79.102982,41.77,MEDIO,NO MANTENER LA DISTANCIA PRUDENCIAL CON RESPEC...,ATROPELLOS,VEHICULO DEPORTIVO UTILITARIO,0.40,0.60


## 16. Resumen

La Fase 3 carga el modelo y el preprocesador, recibe nuevos datos, predice el riesgo, consulta información histórica y genera archivos CSV para Power BI.